# UAH Full Dataset Builder

## Objective

The goal of this notebook is to automatically process every trip in the UAH-DriveSet.

Pipeline:

- Load raw sensor files
- Synchronize sensors
- Engineer new features
- Create sliding windows
- Extract statistical features
- Assign labels
- Merge all trips into a single machine learning dataset

Output:

- final_uah_dataset.csv

In [1]:
import os
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

print(dataset_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1


In [13]:
def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df

def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df

def engineer_features(master_df):

    master_df = master_df.copy()

    # -------------------------------------------------
    # Acceleration Features
    # -------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # -------------------------------------------------
    # Delta Features
    # -------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    return master_df

def extract_statistics(signal):

    features = {}

    features["mean"] = signal.mean()

    features["std"] = signal.std()

    features["min"] = signal.min()

    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal**2)
    )

    return features

def extract_window_features(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics(window[feature])

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

def create_sliding_windows(
        feature_df,
        feature_list,
        window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features(
            window,
            feature_list
        )

        all_window_features.append(window_stats)

    return pd.DataFrame(all_window_features)

def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df

In [3]:
drivers = sorted(

    d
    for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
    and d.startswith("D")

)

print(drivers)

['D1', 'D2', 'D3', 'D4', 'D5', 'D6']


In [4]:
trip_list = []

for driver in drivers:

    driver_path = os.path.join(dataset_path, driver)

    trips = sorted(

        trip
        for trip in os.listdir(driver_path)
        if os.path.isdir(
            os.path.join(driver_path, trip)
        )

    )

    for trip in trips:

        trip_list.append(

            {
                "driver": driver,
                "trip": trip
            }

        )

In [5]:
len(trip_list)

40

In [6]:
trip_list[0]

{'driver': 'D1', 'trip': '20151110175712-16km-D1-NORMAL1-SECONDARY'}

In [7]:
trip_list[-1]

{'driver': 'D6', 'trip': '20151221120051-26km-D6-AGGRESSIVE-MOTORWAY'}

In [8]:
sample_trip = trip_list[0]

sample_trip

{'driver': 'D1', 'trip': '20151110175712-16km-D1-NORMAL1-SECONDARY'}

In [9]:
sample_trip = trip_list[0]

sample_trip

{'driver': 'D1', 'trip': '20151110175712-16km-D1-NORMAL1-SECONDARY'}

In [10]:
trip_path = os.path.join(
    dataset_path,
    sample_trip["driver"],
    sample_trip["trip"]
)

print(trip_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1/D1/20151110175712-16km-D1-NORMAL1-SECONDARY


In [11]:
acc_path = os.path.join(
    trip_path,
    "RAW_ACCELEROMETERS.txt"
)

gps_path = os.path.join(
    trip_path,
    "RAW_GPS.txt"
)

print(acc_path)
print(gps_path)

/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1/D1/20151110175712-16km-D1-NORMAL1-SECONDARY/RAW_ACCELEROMETERS.txt
/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1/D1/20151110175712-16km-D1-NORMAL1-SECONDARY/RAW_GPS.txt


In [14]:
acc_df = load_accelerometer(acc_path)

gps_df = load_gps(gps_path)

In [15]:
def parse_trip_info(trip_name):

    parts = trip_name.split("-")

    return {
        "date": parts[0],
        "distance": parts[1],
        "driver": parts[2],
        "behavior": parts[3],
        "road_type": parts[4]
    }

In [16]:
sample_trip = trip_list[0]["trip"]

info = parse_trip_info(sample_trip)

info

{'date': '20151110175712',
 'distance': '16km',
 'driver': 'D1',
 'behavior': 'NORMAL1',
 'road_type': 'SECONDARY'}

In [18]:
def simplify_behavior(label):

    if "NORMAL" in label:
        return "NORMAL"

    if "AGGRESSIVE" in label:
        return "AGGRESSIVE"

    if "DROWSY" in label:
        return "DROWSY"

    return label

In [19]:
print(simplify_behavior("NORMAL1"))
print(simplify_behavior("NORMAL2"))
print(simplify_behavior("AGGRESSIVE"))
print(simplify_behavior("DROWSY"))

NORMAL
NORMAL
AGGRESSIVE
DROWSY


In [23]:
SAMPLING_RATE = 10      # Hz
WINDOW_SECONDS = 3      # seconds

WINDOW_SIZE = SAMPLING_RATE * WINDOW_SECONDS

window_features = [

    "acc_resultant",

    "acc_horizontal",

    "speed",

    "speed_delta",

    "roll",

    "pitch",

    "yaw"

]

In [21]:
def process_trip(
    dataset_path,
    driver,
    trip
):

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    # Load Sensors
    acc_df = load_accelerometer(acc_path)
    gps_df = load_gps(gps_path)

    # Synchronize
    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    # Feature Engineering
    feature_df = engineer_features(master_df)

    # Sliding Window
    window_dataset = create_sliding_windows(
        feature_df,
        window_features,
        WINDOW_SIZE
    )

    # Labels
    info = parse_trip_info(trip)

    window_dataset["driver"] = info["driver"]
    window_dataset["road_type"] = info["road_type"]
    window_dataset["behavior"] = simplify_behavior(
        info["behavior"]
    )

    return window_dataset

In [24]:
sample_driver = trip_list[0]["driver"]

sample_trip = trip_list[0]["trip"]

sample_dataset = process_trip(
    dataset_path,
    sample_driver,
    sample_trip
)

sample_dataset.head()

,acc_resultant_mean,acc_resultant_std,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_horizontal_mean,acc_horizontal_std,acc_horizontal_min,acc_horizontal_max,...,pitch_rms,yaw_mean,yaw_std,yaw_min,yaw_max,yaw_median,yaw_rms,driver,road_type,behavior
0,0.041557,0.015206,0.019339,0.075226,0.037947,0.044164,0.036259,0.015678,0.012207,0.075166,...,0.021821,0.019100,0.005182,0.011,0.025,0.0215,0.019768,D1,SECONDARY,NORMAL
1,0.041732,0.015063,0.019339,0.075226,0.037947,0.044282,0.036556,0.015447,0.012207,0.075166,...,0.022052,0.019533,0.005111,0.011,0.025,0.0220,0.020169,D1,SECONDARY,NORMAL
2,0.041103,0.015089,0.019339,0.075226,0.036665,0.043698,0.035783,0.015511,0.012207,0.075166,...,0.022413,0.020033,0.005082,0.011,0.027,0.0220,0.020647,D1,SECONDARY,NORMAL
3,0.039805,0.015016,0.019339,0.075226,0.034974,0.042455,0.034164,0.016026,0.005831,0.075166,...,0.022767,0.020667,0.005101,0.011,0.030,0.0220,0.021267,D1,SECONDARY,NORMAL
4,0.040324,0.015078,0.019339,0.075226,0.036665,0.042963,0.035077,0.016086,0.005831,0.075166,...,0.023116,0.021333,0.005101,0.011,0.031,0.0225,0.021915,D1,SECONDARY,NORMAL


In [25]:
all_trip_datasets = []

In [26]:
for trip_info in trip_list:

    print(
        f"Processing: "
        f"{trip_info['driver']} - "
        f"{trip_info['trip']}"
    )

    trip_df = process_trip(

        dataset_path,

        trip_info["driver"],

        trip_info["trip"]

    )

    all_trip_datasets.append(trip_df)

Processing: D1 - 20151110175712-16km-D1-NORMAL1-SECONDARY
Processing: D1 - 20151110180824-16km-D1-NORMAL2-SECONDARY
Processing: D1 - 20151111123124-25km-D1-NORMAL-MOTORWAY
Processing: D1 - 20151111125233-24km-D1-AGGRESSIVE-MOTORWAY
Processing: D1 - 20151111132348-25km-D1-DROWSY-MOTORWAY
Processing: D1 - 20151111134545-16km-D1-AGGRESSIVE-SECONDARY
Processing: D1 - 20151111135612-13km-D1-DROWSY-SECONDARY
Processing: D2 - 20151120131714-26km-D2-NORMAL-MOTORWAY
Processing: D2 - 20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
Processing: D2 - 20151120135152-25km-D2-DROWSY-MOTORWAY
Processing: D2 - 20151120160904-16km-D2-NORMAL1-SECONDARY
Processing: D2 - 20151120162105-17km-D2-NORMAL2-SECONDARY
Processing: D2 - 20151120163350-16km-D2-AGGRESSIVE-SECONDARY
Processing: D2 - 20151120164606-16km-D2-DROWSY-SECONDARY
Processing: D3 - 20151126110502-26km-D3-NORMAL-MOTORWAY
Processing: D3 - 20151126113754-26km-D3-DROWSY-MOTORWAY
Processing: D3 - 20151126124208-16km-D3-NORMAL1-SECONDARY
Processing: D3 - 2

In [27]:
full_dataset = pd.concat(
    all_trip_datasets,
    ignore_index=True
)

full_dataset.shape

(310225, 45)

In [28]:
full_dataset.head()

,acc_resultant_mean,acc_resultant_std,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_horizontal_mean,acc_horizontal_std,acc_horizontal_min,acc_horizontal_max,...,pitch_rms,yaw_mean,yaw_std,yaw_min,yaw_max,yaw_median,yaw_rms,driver,road_type,behavior
0,0.041557,0.015206,0.019339,0.075226,0.037947,0.044164,0.036259,0.015678,0.012207,0.075166,...,0.021821,0.019100,0.005182,0.011,0.025,0.0215,0.019768,D1,SECONDARY,NORMAL
1,0.041732,0.015063,0.019339,0.075226,0.037947,0.044282,0.036556,0.015447,0.012207,0.075166,...,0.022052,0.019533,0.005111,0.011,0.025,0.0220,0.020169,D1,SECONDARY,NORMAL
2,0.041103,0.015089,0.019339,0.075226,0.036665,0.043698,0.035783,0.015511,0.012207,0.075166,...,0.022413,0.020033,0.005082,0.011,0.027,0.0220,0.020647,D1,SECONDARY,NORMAL
3,0.039805,0.015016,0.019339,0.075226,0.034974,0.042455,0.034164,0.016026,0.005831,0.075166,...,0.022767,0.020667,0.005101,0.011,0.030,0.0220,0.021267,D1,SECONDARY,NORMAL
4,0.040324,0.015078,0.019339,0.075226,0.036665,0.042963,0.035077,0.016086,0.005831,0.075166,...,0.023116,0.021333,0.005101,0.011,0.031,0.0225,0.021915,D1,SECONDARY,NORMAL


In [29]:
full_dataset["behavior"].value_counts()

,count
behavior,
NORMAL,131373
DROWSY,99504
AGGRESSIVE,79348


In [30]:
output_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed"

os.makedirs(output_path, exist_ok=True)

full_dataset.to_csv(
    os.path.join(output_path, "uah_full_dataset.csv"),
    index=False
)

print("Full dataset saved successfully!")

Full dataset saved successfully!


In [31]:
saved_df = pd.read_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/uah_full_dataset.csv"
)

saved_df.shape

(310225, 45)